# External Access to Unity Catalog Managed Iceberg Tables

## Overview

This notebook demonstrates how to enable **external open-source Apache Spark clients** to perform full CRUD operations on Unity Catalog–managed Iceberg tables in Databricks. This showcases the power of open standards and vendor-neutral data access.

## What You'll Learn

- 🔧 **External Spark Configuration**: Set up OSS Spark with Iceberg and Unity Catalog integration
- 🔐 **Authentication & Authorization**: Securely connect to UC-managed tables using REST API
- 📊 **Full CRUD Operations**: Create, Read, Update, and Delete data from external clients
- 🌐 **Multi-Platform Data Sharing**: Enable seamless data collaboration across organizations
- 📈 **Real-World Use Case**: Work with comprehensive MLB StatCast baseball analytics data

## Key Benefits

- **Vendor Independence**: No vendor lock-in - use any Spark-compatible platform
- **Centralized Governance**: Unity Catalog maintains metadata, permissions, and audit trails
- **Open Standards**: Built on Apache Iceberg for maximum interoperability
- **Secure Access**: Authentication tokens ensure controlled external access
- **Cost Efficiency**: External compute can access managed storage without data duplication

## Prerequisites

- Databricks workspace with Unity Catalog enabled
- Access token with appropriate permissions
- Python environment with PySpark capabilities
- Unity Catalog managed Iceberg table (we'll use MLB player statistics)


In [1]:
# Install required dependencies for external Spark client
# Uncomment the line below if running in a fresh environment
# %pip install pyspark==3.5.1

# Note: The following packages will be automatically downloaded via Spark packages:
# - org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.2
# - org.apache.iceberg:iceberg-aws-bundle:1.4.2  
# - software.amazon.awssdk:s3:2.20.89
# - software.amazon.awssdk:auth:2.20.89

In [2]:
from config import *

## 1. External Spark Session Configuration

Setting up an external OSS Spark session to connect to Unity Catalog managed Iceberg tables requires:

1. **Iceberg Dependencies**: Runtime libraries for Iceberg table operations
2. **AWS SDK**: For S3 storage access (where Iceberg data files are stored)
3. **Unity Catalog REST API**: Authentication and metadata access
4. **Proper Configuration**: Catalog setup with workspace URI and access token


In [3]:
import os
from pyspark.sql import SparkSession

# Add Iceberg + AWS dependencies for S3 access (without iceberg-aws to avoid constructor issues)
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages "
    "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.2,"
    "org.apache.iceberg:iceberg-aws-bundle:1.4.2,"
    "software.amazon.awssdk:s3:2.20.89,"
    "software.amazon.awssdk:auth:2.20.89 pyspark-shell"
)

# Create Spark session with Unity Catalog configuration
spark = SparkSession.builder \
    .appName("Unity Catalog Iceberg Client") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config(f"spark.sql.catalog.{UC_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{UC_CATALOG_NAME}.type", "rest") \
    .config(f"spark.sql.catalog.{UC_CATALOG_NAME}.uri", f"https://{WORKSPACE_URI}/api/2.1/unity-catalog/iceberg-rest") \
    .config(f"spark.sql.catalog.{UC_CATALOG_NAME}.warehouse", UC_CATALOG_NAME) \
    .config(f"spark.sql.catalog.{UC_CATALOG_NAME}.token", ACCESS_TOKEN) \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.AnonymousAWSCredentialsProvider") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "true") \
    .getOrCreate()

print("Spark session created successfully!")
print(f"Spark version: {spark.version}")
print(f"Configured catalog: {UC_CATALOG_NAME}")


:: loading settings :: url = jar:file:/Users/alexander.booth/miniconda3/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/alexander.booth/.ivy2/cache
The jars for the packages stored in: /Users/alexander.booth/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
software.amazon.awssdk#s3 added as a dependency
software.amazon.awssdk#auth added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-56479014-19db-4d53-8602-6a25e203191b;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.4.2 in central
	found org.apache.iceberg#iceberg-aws-bundle;1.4.2 in central
	found software.amazon.awssdk#s3;2.20.89 in central
	found software.amazon.awssdk#aws-xml-protocol;2.20.89 in central
	found software.amazon.awssdk#aws-query-protocol;2.20.89 in central
	found software.amazon.awssdk#protocol-core;2.20.89 in central
	found software.amazon.awssdk#sdk-core;2.20.89 in central
	found software.amazon.awssdk#annotations;2.20.89 in central
	found so

Spark session created successfully!
Spark version: 3.5.1
Configured catalog: alexander_booth


## 2. Data Exploration - Understanding the MLB Dataset

Before performing CRUD operations, let's explore our Unity Catalog managed Iceberg table containing comprehensive MLB Statcast data. This table includes advanced baseball analytics with 67+ statistical columns covering traditional stats, expected performance metrics, and detailed Statcast measurements.


## Performing CRUD Operations

In [4]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Set catalog and database for Iceberg
iceberg_catalog = "alexander_booth"  # Adjust this to your catalog name
iceberg_db = "iceberg_demo"
table_name = "mlb_batter_stats_2024"

In [5]:
# Top 10 players by home runs
top_hr_query = f"""
SELECT 
    year,
    player_name,
    home_runs,
    batting_average,
    slugging_percentage,
    plate_appearances
FROM {iceberg_catalog}.{iceberg_db}.{table_name}
ORDER BY home_runs DESC
LIMIT 10
"""

print("🏆 Top 10 Home Run Leaders:")
spark.sql(top_hr_query).show()


🏆 Top 10 Home Run Leaders:


+----+------------------+---------+---------------+-------------------+-----------------+
|year|       player_name|home_runs|batting_average|slugging_percentage|plate_appearances|
+----+------------------+---------+---------------+-------------------+-----------------+
|2024|      Judge, Aaron|       58|          0.322|              0.701|              684|
|2024|    Ohtani, Shohei|       54|           0.31|              0.646|              721|
|2024|Santander, Anthony|       44|          0.235|              0.506|              662|
|2024|        Soto, Juan|       41|          0.289|               0.57|              710|
|2024|    Ozuna, Marcell|       39|          0.302|              0.546|              686|
|2024|     Ramírez, José|       39|          0.279|              0.537|              670|
|2024|     Rooker, Brent|       39|          0.293|              0.562|              610|
|2024|   Schwarber, Kyle|       38|          0.248|              0.485|              688|
|2024| Hen

### 3.1 CREATE Operation - Adding New Player Records

Demonstrating INSERT operations from external Spark clients. We'll add synthetic player records with comprehensive StatCast metrics that match the existing schema structure.

In [6]:
# Create sample new player records to demonstrate INSERT with comprehensive stats
# Using realistic values based on the expanded schema
new_players_data = [
    # John Smith - Power hitter profile
    (999001, "Smith, John", 2024, 
     2500, 1500, 85.5, 600, 550, 160, 120, 25, 3, 12, 120, 65, 450,  # Basic stats
     0.291, 0.200, 0.310, 0.491, 0.367, 0.385, 20.0, 10.8,  # Rate stats  
     0.390, 0.295, 0.375, 0.485,  # Expected stats
     92.5, 12.8, 45.2, 15.5, 2.8, 85,  # Quality of contact
     250, 850, 600, 18.5,  # Plate discipline
     75.2, 6.8, 15.2, -2.1, 28.5,  # Advanced metrics
     15.8, 25.4,  # Value metrics
     -0.004, -0.008, 0.006, -0.005),  # Differences
    
    # Mike Johnson - Contact hitter profile  
    (999002, "Johnson, Mike", 2024,
     2000, 1200, 82.3, 480, 420, 125, 95, 20, 2, 8, 95, 45, 380,  # Basic stats
     0.298, 0.155, 0.345, 0.453, 0.355, 0.370, 19.8, 9.4,  # Rate stats
     0.375, 0.285, 0.350, 0.445,  # Expected stats
     90.1, 11.5, 42.8, 12.2, 2.1, 68,  # Quality of contact
     180, 720, 520, 15.2,  # Plate discipline
     72.8, 6.5, 12.8, 1.8, 25.2,  # Advanced metrics
     12.1, 18.7,  # Value metrics
     0.013, 0.005, 0.008, -0.005),  # Differences
    
    # Dave Williams - Balanced profile
    (999003, "Williams, Dave", 2024,
     1600, 800, 78.9, 320, 290, 85, 65, 12, 1, 7, 75, 30, 250,  # Basic stats
     0.293, 0.172, 0.335, 0.469, 0.350, 0.375, 23.4, 9.4,  # Rate stats
     0.380, 0.285, 0.345, 0.465,  # Expected stats
     91.8, 13.2, 44.5, 13.8, 2.5, 72,  # Quality of contact
     165, 580, 415, 16.8,  # Plate discipline
     73.5, 6.7, 14.1, 0.5, 26.8,  # Advanced metrics
     8.9, 14.2,  # Value metrics
     0.008, 0.005, 0.004, -0.005)  # Differences
]

# Create comprehensive schema matching our expanded dataset
new_players_columns = [
    "player_id", "player_name", "year",
    "total_pitches_seen", "total_pitches", "pitch_percent", "plate_appearances", "at_bats", 
    "hits", "singles", "doubles", "triples", "home_runs", "strikeouts", "walks", "balls_in_play",
    "batting_average", "isolated_power", "babip", "slugging_percentage", "on_base_percentage", 
    "weighted_on_base_avg", "strikeout_rate", "walk_rate",
    "expected_woba", "expected_batting_avg", "expected_obp", "expected_slg",
    "avg_exit_velocity", "avg_launch_angle", "hard_hit_percent", "barrel_rate", "barrels_per_pa", "total_barrels",
    "whiffs", "swings", "takes", "swing_miss_percent",
    "bat_speed", "swing_length", "attack_angle", "attack_direction", "swing_path_tilt",
    "run_expectancy", "batting_run_value",
    "ba_vs_expected_diff", "obp_vs_expected_diff", "slg_vs_expected_diff", "woba_vs_expected_diff"
]

new_players_df = spark.createDataFrame(new_players_data, new_players_columns) \
    .withColumn("last_updated", current_timestamp())

# List of columns that need to be cast from long to int
int_columns = [
    "year", "total_pitches_seen", "total_pitches", "plate_appearances", "at_bats",
    "hits", "singles", "doubles", "triples", "home_runs", "strikeouts", "walks",
    "balls_in_play", "total_barrels", "whiffs", "swings", "takes"
]

# Cast columns to IntegerType
for c in int_columns:
    new_players_df = new_players_df.withColumn(c, col(c).cast("integer"))

print("📝 Created new player records with comprehensive Statcast metrics:")
print("Key stats preview:")
new_players_df.select("player_name", "home_runs", "batting_average", "avg_exit_velocity", 
                     "barrel_rate", "expected_woba", "bat_speed").show()


📝 Created new player records with comprehensive Statcast metrics:
Key stats preview:


25/08/20 13:19:40 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------+---------+---------------+-----------------+-----------+-------------+---------+
|   player_name|home_runs|batting_average|avg_exit_velocity|barrel_rate|expected_woba|bat_speed|
+--------------+---------+---------------+-----------------+-----------+-------------+---------+
|   Smith, John|       12|          0.291|             92.5|       15.5|         0.39|     75.2|
| Johnson, Mike|        8|          0.298|             90.1|       12.2|        0.375|     72.8|
|Williams, Dave|        7|          0.293|             91.8|       13.8|         0.38|     73.5|
+--------------+---------+---------------+-----------------+-----------+-------------+---------+



In [7]:
# INSERT new players into the Iceberg table
new_players_df.writeTo(f"{iceberg_catalog}.{iceberg_db}.{table_name}") \
    .append()

print("✅ Successfully added new players to the Iceberg table")

# Verify the insert by checking the count
total_count = spark.sql(f"SELECT COUNT(*) as count FROM {iceberg_catalog}.{iceberg_db}.{table_name}").collect()[0]["count"]

print(f"📊 Total players in table: {total_count}")


✅ Successfully added new players to the Iceberg table
📊 Total players in table: 654


### 3.2 READ Operation - Verifying Data Access

Confirming that our external client can successfully read data from the Unity Catalog managed Iceberg table, including the newly inserted records.

In [8]:
# Verify our new players were added
verify_query = f"""
SELECT player_name, player_id, home_runs, batting_average, last_updated
FROM {iceberg_catalog}.{iceberg_db}.{table_name}
WHERE player_id >= 999001
ORDER BY player_id
"""

print("🔍 Verifying newly added players:")
spark.sql(verify_query).show()


🔍 Verifying newly added players:
+--------------+---------+---------+---------------+--------------------+
|   player_name|player_id|home_runs|batting_average|        last_updated|
+--------------+---------+---------+---------------+--------------------+
|   Smith, John|   999001|       12|          0.291|2025-08-20 13:19:...|
| Johnson, Mike|   999002|        8|          0.298|2025-08-20 13:19:...|
|Williams, Dave|   999003|        7|          0.293|2025-08-20 13:19:...|
+--------------+---------+---------+---------------+--------------------+



### 3.3 UPDATE Operation - Modifying Existing Records

Demonstrating how external clients can modify existing data in Unity Catalog managed Iceberg tables. We'll update player statistics and recalculate derived metrics.

In [9]:
# Show current stats for a player before update
player_before = spark.sql(f"""
SELECT player_name, home_runs, hits, batting_average, last_updated
FROM {iceberg_catalog}.{iceberg_db}.{table_name}
WHERE player_id = 999001
""")

print("📋 Player stats BEFORE update:")
player_before.show()


📋 Player stats BEFORE update:
+-----------+---------+----+---------------+--------------------+
|player_name|home_runs|hits|batting_average|        last_updated|
+-----------+---------+----+---------------+--------------------+
|Smith, John|       12| 160|          0.291|2025-08-20 13:19:...|
+-----------+---------+----+---------------+--------------------+



In [10]:
# UPDATE operation - Let's say John Smith hit 3 more home runs and got 10 more hits
# We'll update his stats and recalculate batting average
update_query = f"""
UPDATE {iceberg_catalog}.{iceberg_db}.{table_name}
SET 
    home_runs = home_runs + 3,
    hits = hits + 10,
    batting_average = ROUND((hits + 10) / CAST(at_bats AS DOUBLE), 3),
    last_updated = current_timestamp()
WHERE player_id = 999001
"""

spark.sql(update_query)
print("✅ Updated John Smith's stats (added 3 HRs and 10 hits)")


✅ Updated John Smith's stats (added 3 HRs and 10 hits)


In [11]:
# Verify the update
player_after = spark.sql(f"""
SELECT player_name, home_runs, hits, batting_average, last_updated
FROM {iceberg_catalog}.{iceberg_db}.{table_name}
WHERE player_id = 999001
""")

print("📋 Player stats AFTER update:")
player_after.show()


📋 Player stats AFTER update:
+-----------+---------+----+---------------+--------------------+
|player_name|home_runs|hits|batting_average|        last_updated|
+-----------+---------+----+---------------+--------------------+
|Smith, John|       15| 170|          0.309|2025-08-20 13:19:...|
+-----------+---------+----+---------------+--------------------+



### 3.4 DELETE Operation - Removing Records

Demonstrating how external clients can delete data from Unity Catalog managed Iceberg tables. We'll clean up our test records to maintain data integrity.

In [12]:
# Show current test players before deletion
before_delete = spark.sql(f"""
SELECT COUNT(*) as count
FROM {iceberg_catalog}.{iceberg_db}.{table_name}
WHERE player_id >= 999001
""").collect()[0]["count"]

print(f"📊 Test players before deletion: {before_delete}")

📊 Test players before deletion: 3


In [13]:
# DELETE a specific player (let's remove Dave Williams)
delete_query = f"""
DELETE FROM {iceberg_catalog}.{iceberg_db}.{table_name}
WHERE player_id >= 999001
"""

spark.sql(delete_query)
print("✅ Deleted test players")


✅ Deleted test players


In [14]:
# Verify the deletion
after_single_delete = spark.sql(f"""
SELECT player_name, player_id
FROM {iceberg_catalog}.{iceberg_db}.{table_name}
WHERE player_id >= 999001
ORDER BY player_id
""")

print("📋 Remaining test players after single deletion:")
after_single_delete.show()


📋 Remaining test players after single deletion:
+-----------+---------+
|player_name|player_id|
+-----------+---------+
+-----------+---------+



## 4. Session Cleanup

In [15]:
# Stop the Spark session when done
spark.stop()
print("Spark session stopped successfully!")


Spark session stopped successfully!


## ✅ Summary & Key Achievements

This notebook successfully demonstrates **vendor-neutral data access** to Unity Catalog managed Iceberg tables using external open-source Spark clients.

### 🎯 What We Accomplished

1. **External Client Setup**: Configured OSS Spark with Iceberg dependencies and Unity Catalog integration
2. **Full CRUD Operations**: Successfully performed Create, Read, Update, and Delete operations from outside Databricks
3. **Authentication**: Securely connected using access tokens and REST API endpoints
4. **Data Integrity**: Maintained consistency across all operations with ACID transaction guarantees
5. **Real-World Data**: Worked with comprehensive MLB StatCast analytics (67+ statistical columns)

### 🏗️ Architecture Benefits

- **🔓 No Vendor Lock-in**: Use any Spark-compatible platform (EMR, Dataproc, on-premises, etc.)
- **🎯 Centralized Governance**: Unity Catalog maintains metadata, permissions, lineage, and audit trails
- **📈 Cost Optimization**: External compute can access managed storage without data duplication
- **🌐 Open Standards**: Built on Apache Iceberg for maximum ecosystem compatibility
- **🔐 Security**: Token-based authentication ensures controlled external access

### 🚀 Use Cases Enabled

- **Partner Data Sharing**: Secure data collaboration across organizations
- **Multi-Cloud Analytics**: Access Databricks-managed data from other cloud platforms  
- **Legacy System Integration**: Connect existing Spark workloads to Unity Catalog
- **Cost-Effective Compute**: Use lower-cost external compute for certain workloads
- **Disaster Recovery**: External access ensures business continuity

### 🔧 Technical Implementation

- **REST API**: Unity Catalog Iceberg REST interface for metadata operations
- **S3 Storage**: Direct access to Iceberg data files in cloud storage
- **ACID Transactions**: Full consistency guarantees across all operations
- **Schema Evolution**: Support for evolving table schemas over time

This architecture enables **secure, governed, and cost-effective data collaboration** across any Spark ecosystem while maintaining centralized control through Unity Catalog.